# Extremal Modelling Results

Results of the gbex modelling for the extreme snowfall and extreme temperature anomalies, including final model fits, metrics, and interpretation metrics.

Jimmy Butler, 11/2025

In [5]:
library(gbex)
library(tidyverse)
library(ggplot2)
library(here)
library(grf)
library(jsonlite)
source('interpretation_utils.R')

root_dir <- here()

## Snowfall

### Loading Data

In [95]:
train_dat_full <- read_csv(paste0(root_dir, '/project/dataset/datasets/model_ready/train.csv'))
test_dat_full <- read_csv(paste0(root_dir, '/project/dataset/datasets/model_ready/test.csv'))

feature_cols <- c('cumulative_landfalling_area', 'max_south_extent', 'max_IWV_ais', 'max_ocean_SLP_gradient', 'max_landfalling_v850hPa', 'avg_landfalling_minomega')
X_train <- train_dat_full %>% select(feature_cols)
y_train_snow <- train_dat_full %>% select('cumulative_snowfall_ais') %>% pull()
y_train_temp <- train_dat_full %>% select('max_T2M_anomaly_ais') %>% pull()

Rows: 2481 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): Label
dbl (22): max_area, mean_area, mean_landfalling_area, cumulative_landfalling...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 620 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): Label
dbl (22): max_area, mean_area, mean_landfalling_area, cumulative_landfalling...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Fitting the Intermediate Model

In [96]:
tau_0 <- 0.7 #lower from 0.8 because otherwise only 60 exceedances!
grf_param_path <- paste0(root_dir, '/project/modelling/gbex/cross_validation/best_snow_grf.json')
best_grf_params <- fromJSON(grf_param_path)

fit_snow_threshold <- quantile_forest(
  X_train, y_train_snow,
  quantiles = tau_0,
  num.trees = best_grf_params$num_trees,
  min.node.size = best_grf_params$min_node_size,
  sample.fraction = best_grf_params$sample_frac,
  alpha = best_grf_params$alpha,
  honesty = FALSE,
  seed = 54321
)

Now, we grab the exceedances for the training dataset.

In [97]:
u_train <- predict(fit_snow_threshold, X_train, quantiles = tau_0)$predictions[,1]
diffs <- y_train_snow - u_train
z_train <- diffs[diffs > 0] # only take exceedances above the intermediate threshold
X_train_gbex <- X_train[diffs > 0, ] # take the corresponding covariates as well

In [98]:
write_csv(X_train_gbex, 'plotting_data/gbex_snow_X.csv')

### Fitting the Extreme Model

First, we load up the hyperparameters from the best-fitting model, determined via grid search and 5-fold CV.

In [99]:
gbex_param_path <- paste0(root_dir, '/project/modelling/gbex/cross_validation/best_snow_gbex.json')
best_gbex_params <- fromJSON(gbex_param_path)

In [100]:
gbex_snow <- gbex(y=z_train,
         X=X_train_gbex, 
         B=best_gbex_params$num_trees,
         lambda_scale=best_gbex_params$lambda_scale,
         lambda_ratio=best_gbex_params$lambda_ratio,
         depth=c(best_gbex_params$depth_sigma, best_gbex_params$depth_gamma),
         min_leaf_size=c(best_gbex_params$min_leaf_sigma, best_gbex_params$min_leaf_gamma),
         sf=best_gbex_params$sf)

Fit gbex
  |============================================================================| 100%


### Evaluating and Interpreting Extreme Model

Let's first grab the test data..

In [101]:
X_test <- test_dat_full %>% select(feature_cols)
y_test_snow <- test_dat_full %>% select('cumulative_snowfall_ais') %>% pull()
y_test_temp <- test_dat_full %>% select('max_T2M_anomaly_ais') %>% pull()

u_test <- predict(fit_snow_threshold, X_test, quantiles = tau_0)$predictions[,1]
diffs <- y_test_snow - u_test
z_test <- diffs[diffs > 0] # only take exceedances above the intermediate threshold
X_test_gbex <- X_test[diffs > 0, ] # take the corresponding covariates as well

In [102]:
test_preds <- predict(gbex_snow, X_test_gbex)
test_preds$z_test <- z_test

In [103]:
mean(apply(test_preds, 1, GP_dev))

[1] -1.243332

### Permutation Scores

In [105]:
# concatenate training and testing data to evaluate model diagnostics
X_full_gbex <- rbind(X_train_gbex, X_test_gbex)
z_full_gbex <- c(z_train, z_test)
write_csv(X_full_gbex, 'plotting_data/gbex_snow_fullX.csv')

In [18]:
permutation_scores <- permutation_score(gbex_snow, data.frame(X_full_gbex), z_full_gbex, n_reps=500)

In [21]:
write_csv(permutation_scores, 'plotting_data/snow_gbex_VI.csv')

### 1D Partial Dependence Plots

In [56]:
# the extreme quantile we wish to look at
tau <- 0.995
# the quantile used to clip extreme observations in feature space
q <- 0.1

In [57]:
var_name <- 'max_IWV_ais'
extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_snow, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/IWV_995_snow.csv')

In [58]:
var_name <- 'max_landfalling_v850hPa'

extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_snow, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/wind_995_snow.csv')

In [59]:
var_name <- 'avg_landfalling_minomega'

extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_snow, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/omega_995_snow.csv')

In [60]:
var_name <- 'cumulative_landfalling_area'

extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_snow, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/CLA_995_snow.csv')

### 2D Partial Dependence Plots

In [32]:
# the extreme quantile we wish to look at
tau <- 0.995
# the quantile used to clip extreme observations in feature space
q <- 0.1

In [62]:
var_names <- c('max_IWV_ais', 'max_landfalling_v850hPa')
extents_x <- quantile(X_full_gbex[[var_names[1]]], c(q, 1-q))
extents_y <- quantile(X_full_gbex[[var_names[2]]], c(q, 1-q))
extents <- c(extents_x[[1]], extents_x[[2]], extents_y[[1]], extents_y[[2]])
plt_df <- compute_quantile_PDP_2D(gbex_snow, tau, X_full_gbex, var_names, 20, extents)
write_csv(plt_df$PD_df, 'plotting_data/IWV_wind_995_snow.csv')

In [63]:
var_names <- c('max_IWV_ais', 'avg_landfalling_minomega')
extents_x <- quantile(X_full_gbex[[var_names[1]]], c(q, 1-q))
extents_y <- quantile(X_full_gbex[[var_names[2]]], c(q, 1-q))
extents <- c(extents_x[[1]], extents_x[[2]], extents_y[[1]], extents_y[[2]])
plt_df <- compute_quantile_PDP_2D(gbex_snow, tau, X_full_gbex, var_names, 20, extents)
write_csv(plt_df$PD_df, 'plotting_data/IWV_omega_995_snow.csv')

In [64]:
var_names <- c('max_IWV_ais', 'cumulative_landfalling_area')
extents_x <- quantile(X_full_gbex[[var_names[1]]], c(q, 1-q))
extents_y <- quantile(X_full_gbex[[var_names[2]]], c(q, 1-q))
extents <- c(extents_x[[1]], extents_x[[2]], extents_y[[1]], extents_y[[2]])
plt_df <- compute_quantile_PDP_2D(gbex_snow, tau, X_full_gbex, var_names, 20, extents)
write_csv(plt_df$PD_df, 'plotting_data/IWV_CLA_995_snow.csv')

## Temperature Anomaly

In [106]:
tau_0 <- 0.7 #lower from 0.8 because otherwise only 60 exceedances!
grf_param_path <- paste0(root_dir, '/project/modelling/gbex/cross_validation/best_temp_grf.json')
best_grf_params <- fromJSON(grf_param_path)

fit_temp_threshold <- quantile_forest(
  X_train, y_train_temp,
  quantiles = tau_0,
  num.trees = best_grf_params$num_trees,
  min.node.size = best_grf_params$min_node_size,
  sample.fraction = best_grf_params$sample_frac,
  alpha = best_grf_params$alpha,
  honesty = FALSE,
  seed = 54321
)

In [108]:
u_train <- predict(fit_temp_threshold, X_train, quantiles = tau_0)$predictions[,1]
diffs <- y_train_temp - u_train
z_train <- diffs[diffs > 0] # only take exceedances above the intermediate threshold
X_train_gbex <- X_train[diffs > 0, ] # take the corresponding covariates as well

In [109]:
write_csv(X_train_gbex, 'plotting_data/gbex_temp_X.csv')

### Fitting the Extreme Model

Let's first grab the test data.

In [110]:
gbex_param_path <- paste0(root_dir, '/project/modelling/gbex/cross_validation/best_temp_gbex.json')
best_gbex_params <- fromJSON(gbex_param_path)

In [111]:
gbex_temp <- gbex(y=z_train,
         X=X_train_gbex, 
         B=best_gbex_params$num_trees,
         lambda_scale=best_gbex_params$lambda_scale,
         lambda_ratio=best_gbex_params$lambda_ratio,
         depth=c(best_gbex_params$depth_sigma, best_gbex_params$depth_gamma),
         min_leaf_size=c(best_gbex_params$min_leaf_sigma, best_gbex_params$min_leaf_gamma),
         sf=best_gbex_params$sf)

Fit gbex
  |                                                                            |   0%

Warning message in first_guess(y, gamma_positive):
“Unconditional estimate of gamma is negative (initial estimate is set to 0.01”


  |============================================================================| 100%


In [112]:
X_test <- test_dat_full %>% select(feature_cols)
y_test_temp <- test_dat_full %>% select('max_T2M_anomaly_ais') %>% pull()

u_test <- predict(fit_temp_threshold, X_test, quantiles = tau_0)$predictions[,1]
diffs <- y_test_temp - u_test
z_test <- diffs[diffs > 0] # only take exceedances above the intermediate threshold
X_test_gbex <- X_test[diffs > 0, ] # take the corresponding covariates as well

In [113]:
test_preds <- predict(gbex_temp, X_test_gbex)
test_preds$z_test <- z_test

In [114]:
mean(apply(test_preds, 1, GP_dev))

[1] 2.40851

### Permutation Scores

In [115]:
# concatenate training and testing data to evaluate model diagnostics
X_full_gbex <- rbind(X_train_gbex, X_test_gbex)
z_full_gbex <- c(z_train, z_test)
write_csv(X_full_gbex, 'plotting_data/gbex_temp_fullX.csv')

In [79]:
permutation_scores <- permutation_score(gbex_temp, data.frame(X_full_gbex), z_full_gbex, n_reps=500)

In [80]:
write_csv(permutation_scores, 'plotting_data/temp_gbex_VI.csv')

### 1D Partial Dependence Plots

In [ ]:
# the extreme quantile we wish to look at
tau <- 0.995
# the quantile used to clip extreme observations in feature space
q <- 0.1

In [118]:
var_name <- 'max_IWV_ais'
extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_temp, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/IWV_995_temp.csv')

In [116]:
var_name <- 'max_landfalling_v850hPa'
extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_temp, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/wind_995_temp.csv')

In [83]:
var_name <- 'avg_landfalling_minomega'
extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_temp, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/omega_995_temp.csv')

In [84]:
var_name <- 'cumulative_landfalling_area'
extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_temp, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/CLA_995_temp.csv')

In [86]:
var_name <- 'max_south_extent'
extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_temp, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/south_995_temp.csv')

In [89]:
var_name <- 'max_ocean_SLP_gradient'
extents_x <- quantile(X_full_gbex[[var_name]], c(q, 1-q))
plt_df <- compute_quantile_PDP_1D(gbex_temp, tau, X_full_gbex, var_name, extents=c(extents_x[[1]], extents_x[[2]]))
write_csv(plt_df$PD_df, 'plotting_data/SLP_995_temp.csv')

### 2D Partial Dependence Plots

In [70]:
# the extreme quantile we wish to look at
tau <- 0.995
# the quantile used to clip extreme observations in feature space
q <- 0.1

In [92]:
var_names <- c('max_IWV_ais', 'max_landfalling_v850hPa')
extents_x <- quantile(X_full_gbex[[var_names[1]]], c(q, 1-q))
extents_y <- quantile(X_full_gbex[[var_names[2]]], c(q, 1-q))
extents <- c(extents_x[[1]], extents_x[[2]], extents_y[[1]], extents_y[[2]])
plt_df <- compute_quantile_PDP_2D(gbex_temp, tau, X_full_gbex, var_names, 20, extents)
write_csv(plt_df$PD_df, 'plotting_data/IWV_wind_995_temp.csv')

In [93]:
var_names <- c('max_south_extent', 'max_landfalling_v850hPa')
extents_x <- quantile(X_full_gbex[[var_names[1]]], c(q, 1-q))
extents_y <- quantile(X_full_gbex[[var_names[2]]], c(q, 1-q))
extents <- c(extents_x[[1]], extents_x[[2]], extents_y[[1]], extents_y[[2]])
plt_df <- compute_quantile_PDP_2D(gbex_temp, tau, X_full_gbex, var_names, 20, extents)
write_csv(plt_df$PD_df, 'plotting_data/south_wind_995_temp.csv')

In [94]:
var_names <- c('max_ocean_SLP_gradient', 'max_landfalling_v850hPa')
extents_x <- quantile(X_full_gbex[[var_names[1]]], c(q, 1-q))
extents_y <- quantile(X_full_gbex[[var_names[2]]], c(q, 1-q))
extents <- c(extents_x[[1]], extents_x[[2]], extents_y[[1]], extents_y[[2]])
plt_df <- compute_quantile_PDP_2D(gbex_temp, tau, X_full_gbex, var_names, 20, extents)
write_csv(plt_df$PD_df, 'plotting_data/SLP_wind_995_temp.csv')